In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import statsmodels.api as sm
import scipy.stats as stats
from scipy.stats import contingency
#from sklearn.preprocessing import StandardScaler
#from sklearn.metrics import r2_score
#from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

### Datenaufbereitung

Lade den Datensatz über die verfügbaren Apps im Google Play Store (googleplaystore.csv).

Überprüfe die Datenqualität anhand der Merkmale *Type* und *Price*. Gibt es Apps, die im Merkmal *Type* eine Merkmalsausprägung *Free* haben, aber einen Preis ungleich Null? Ebenso soll überprüft werden, ob es Apps mit *Price = 0* gibt, deren *Type* von *Free* abweicht. Einträge, die diese Kriterien verletzen, sollen aus dem Datensatz entfernt werden.

Überprüfe abschließend, ob Duplikate in den Einträgen existieren, und entferne diese gegebenenfalls.

In [28]:
apps = pd.read_csv('googleplaystore.csv', index_col=0)
#display(apps.head(5))
#display(apps.tail(5))
print('Größe des Datensatzes: {}'.format(apps.shape))

#Typ Free aber Preis ungleich 0
apps_Type_Free_Price=apps[
    (apps['Type']=='Free')&
    (apps['Price']!='0')
]
#display(apps_Type_Free_Price.head(5))
print('Größe des Datensatzes nach Suche nach Type "Free" und Preis ungleich "0": {}'.format(apps_Type_Free_Price.shape))

#Preis = 0 aber Typ ungleich Free
apps_Price_0_Type=apps[
    (apps['Price']=='0')&
    (apps['Type']!='Free')   
]
#display(apps_Price_0_Type.head(5))
print('Größe des Datensatzes nach Suche nach Type "Free" und Preis ungleich "0": {}'.format(apps_Price_0_Type.shape))

#Löschen der fehlerhaften Einträge

idx_bad_1 = apps_Type_Free_Price.index
idx_bad_2 = apps_Price_0_Type.index
idx_all_bad = idx_bad_1.union(idx_bad_2)

if not idx_all_bad.empty:
    apps_cleaned = apps.drop(idx_all_bad)
    print(f"{len(idx_all_bad)} fehlerhafte Einträge wurden entfernt.")
else:
    apps_cleaned = apps.copy()
    print("Keine fehlerhaften Einträge gefunden – es wurde nichts gelöscht.")

print("Größe nach dem Löschen:", apps_cleaned.shape)

#Duplikate
dup_count = apps_cleaned.duplicated().sum()
print(f'Anzahl der Duplikate im bereinigten Datensatz: {dup_count}')

if dup_count > 0:
    apps_cleaned_duplicate_free = apps_cleaned.drop_duplicates(keep='first')
    print(f'{dup_count} Duplikate wurden entfernt (1 Eintrag pro Duplikat bleibt erhalten).')
else:
    apps_cleaned_duplicate_free = apps_cleaned.copy()
    print('Keine Duplikate gefunden.')

print('Endgröße nach vollständiger Bereinigung:', apps_cleaned_duplicate_free.shape)


Größe des Datensatzes: (10841, 12)
Größe des Datensatzes nach Suche nach Type "Free" und Preis ungleich "0": (0, 12)
Größe des Datensatzes nach Suche nach Type "Free" und Preis ungleich "0": (1, 12)
1 fehlerhafte Einträge wurden entfernt.
Größe nach dem Löschen: (10840, 12)
Anzahl der Duplikate im bereinigten Datensatz: 492
492 Duplikate wurden entfernt (1 Eintrag pro Duplikat bleibt erhalten).
Endgröße nach vollständiger Bereinigung: (10348, 12)


### Datenuntersuchung

Gib folgende Kennzahlen und Eigenschaften des Datensatzes an:  
- Wie viele Einträge existieren?  
- Auf welchen Skalen sind die unterschiedlichen Merkmale definiert?  
- Welchen Modus haben die Merkmale *Category*, *Content Rating* und *Genres*?  

Stelle zusätzlich folgende Grafiken dar:  
- für *Category*: absolute Häufigkeit  
- für *Content Rating*: relative Häufigkeit  
- für *Genres*: absolute Häufigkeit der Top 20, sortiert nach Anzahl  

Wie viele Merkmalsausprägungen existieren für *Genres*?

### Zusammenhang Qualitative Merkmale

Erstelle die Kreuztabelle für die Merkmale *Content Rating* und *Type*. Berechne damit die folgenden bedingten Häufigkeiten:

- P(Paid | Everyone)  
- P(Mature | Free)  
- Die Wahrscheinlichkeit, dass eine App mit Content Rating *Teen* kostenlos ist.  

Bewerte anschließend die statistische Abhängigkeit der beiden Merkmale basierend auf dem korrigierten Kontingenzkoeffizienten.  


### Quantitative Merkmale

Lade den Datensatz *Advertising.csv* ein.

Der Datensatz hat 4 Merkmale und 200 Beobachtungen. Die Merkmale sind:
- *TV* - Werbebudget in Geldeinheiten [GE] für Spots im TV
- *radio* - Werbebudget in GE für Ansagen im Radio
- *newspaper* - Werbebudget in GE für Announcen in Zeitungen
- *sales* - Anzahl verkaufter Produkte [in Tsd.].

Fragestellungen:  
- Stelle für das Merkmal *TV* einen kumulativen Häufigkeitsplot dar. Welches minimale Budget haben die 30% Kampagnen mit dem höchsten TV-Budget?  
- Stelle für das Merkmal *newspaper* einen Boxplot dar. Wie viele Ausreißer existieren?

Gib zusätzlich für alle Merkmale jeweils den Median, den Interquartilsabstand, den arithmetischen Mittelwert und die Standardabweichung an.  

### Zusammenhänge quantitativer Merkmale

Stelle einen pairplot für den Datensatz dar und bestimme die Korrelationsmatrix.

Berechne die Lineare Regression für *sales* basierend auf *TV* und beantworte folgende Fragen:
- Wie lautet das Modell der Linearen Regression?
- Sind die Koeffizienten signifikant?
- Welche Aussagen über die Residuen lassen ableiten?
- Welche Aussage lässt sich aus dem Bestimmtsmaß ableiten?  
- Würdest du das Modell in der Praxis benutzen?

Stelle zusätzlich Grafiken dar, um das Ergebnis und die Qualität der Linearen Regression zu visualisieren.


Bestimme danach die multiple lineare Regression für *sales* bei Benutzung der aller Merkmale.

Beantworte folgende Fragen:

- Ist die multiple lineare Regression *besser* als die einfache lineare Regression?
- Haben alle 3 Merkmale einen Einfluss auf das Ergebnis?

Stelle auch für diesen Fall Grafiken dar, um das Ergebnis und die Qualität zu visualisieren. 

In [ ]:
# Datenaufbereitung für googleplaystore.csv
import pandas as pd
import numpy as np
import re

# Pfad zur Datei ggf. anpassen
path = 'googleplaystore.csv'
df = pd.read_csv(path)
print('Ursprüngliche Form:', df.shape)

# Normalisierung der Spalten
if 'Type' in df.columns:
    df['Type'] = df['Type'].astype(str).str.strip().str.capitalize()
else:
    print("Spalte 'Type' nicht gefunden")

# Funktion zum Parsen des Price-Feldes
def parse_price(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.lower() == 'free':
        return 0.0
    # entferne $ und alles außer Ziffern und Punkt
    s = re.sub(r'[^0-9\.]','',s)
    try:
        return float(s)
    except Exception:
        return np.nan

df['Price_num'] = df['Price'].apply(parse_price) if 'Price' in df.columns else np.nan
print('Beispiele Price_orig -> Price_num:')
display(df[['Price']].dropna().head(10))
display(df[['Price_num']].head(10))

# Inkonsequente Einträge finden
mask_type_free_but_price_pos = (df['Type'].str.lower() == 'free') & (df['Price_num'] > 0)
mask_price_zero_but_type_not_free = (df['Price_num'] == 0) & (df['Type'].str.lower() != 'free')
print('Type=Free aber Price>0  :', mask_type_free_but_price_pos.sum())
print('Price=0 aber Type!=Free :', mask_price_zero_but_type_not_free.sum())

# Zeige Beispiele, falls vorhanden
if mask_type_free_but_price_pos.sum() > 0:
    print('\nBeispiele Type=Free aber Price>0:')
    display(df.loc[mask_type_free_but_price_pos].head(10))
if mask_price_zero_but_type_not_free.sum() > 0:
    print('\nBeispiele Price=0 aber Type!=Free:')
    display(df.loc[mask_price_zero_but_type_not_free].head(10))

# Entferne die verletzenden Einträge
mask_inconsistent = mask_type_free_but_price_pos | mask_price_zero_but_type_not_free
df_clean = df.loc[~mask_inconsistent].copy()
print('Nach Entfernen inkonsistenter Zeilen:', df_clean.shape)

# Duplikate prüfen
dups_all = df_clean.duplicated().sum()
dups_name = df_clean.duplicated(subset=['App']).sum() if 'App' in df_clean.columns else 0
print('Anzahl exakte Duplikate (ganze Zeile):', dups_all)
print("Anzahl Duplikate nach 'App':", dups_name)

# Entfernen der Duplikate (ganze Zeile und dann nach App)
df_clean = df_clean.drop_duplicates()
if 'App' in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=['App'], keep='first')
print('Nach Entfernen von Duplikaten:', df_clean.shape)

# Speichern
out = 'googleplaystore_clean.csv'
df_clean.to_csv(out, index=False)
print('Bereinigter Datensatz gespeichert als:', out)

# Zusammenfassung
print('\nZusammenfassung:')
print('Ursprünglich:', df.shape)
print('Bereinigt :', df_clean.shape)
print('Entfernte Zeilen gesamt:', df.shape[0]-df_clean.shape[0])